# 08 · Stress Testing

## Credit Risk Intelligence Lab

This notebook closes the first complete version of the **Credit Risk Intelligence Lab** by adding a stress testing and scenario analysis layer.

The previous notebooks built the analytical foundation of the project:

1. Exploratory credit risk analysis.
2. Feature engineering and preprocessing.
3. Probability of Default model.
4. Unsupervised risk segmentation.
5. Anomaly detection.
6. Model interpretability.
7. Portfolio-level risk reporting.

This notebook asks a final portfolio risk question: **what happens to expected loss if credit conditions deteriorate?**

In credit risk management, a model estimate is only one view of risk. A portfolio should also be examined under adverse scenarios. Stress testing helps evaluate sensitivity to increases in probability of default, loss given default, exposure, and segment-specific deterioration.

## 1. Notebook objectives

This notebook focuses on eight tasks:

1. Load the consolidated portfolio risk report.
2. Define base, mild, adverse, and severe stress scenarios.
3. Apply shocks to PD, LGD, and EAD.
4. Add extra stress to high-risk clusters, high-risk bands, and anomalous borrowers.
5. Calculate stressed expected loss under each scenario.
6. Compare portfolio-level risk across scenarios.
7. Analyze stress impact by risk band, cluster, and anomaly flag.
8. Save stress testing tables, figures, metadata, and a markdown stress report.

This notebook is not intended to be a regulatory stress testing framework. It is an analytical simulation layer that demonstrates how credit portfolio risk can be evaluated under alternative assumptions.

In [ ]:
from pathlib import Path
import warnings
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 240)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 2. Project paths

The notebook is designed to run either from the project root or from the `notebooks/` directory.

In [ ]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_FIGURES = REPORTS_DIR / "figures"
REPORTS_TABLES = REPORTS_DIR / "tables"
MODELS_DIR = PROJECT_ROOT / "models"

for path in [DATA_PROCESSED, REPORTS_DIR, REPORTS_FIGURES, REPORTS_TABLES, MODELS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed data path: {DATA_PROCESSED}")
print(f"Reports path: {REPORTS_DIR}")
print(f"Figures path: {REPORTS_FIGURES}")
print(f"Tables path: {REPORTS_TABLES}")

## 3. Load consolidated portfolio report

The preferred input is the output from notebook 07:

```text
data/processed/credit_risk_portfolio_report.csv
```

If that file is missing, the notebook falls back to:

```text
data/processed/credit_risk_segments.csv
```

The stress testing layer requires, at minimum, borrower-level predicted PD, exposure proxy, actual default label, and expected loss proxy.

In [ ]:
portfolio_report_file = DATA_PROCESSED / "credit_risk_portfolio_report.csv"
segments_file = DATA_PROCESSED / "credit_risk_segments.csv"

if portfolio_report_file.exists():
    portfolio = pd.read_csv(portfolio_report_file, index_col=0)
    print("Loaded consolidated portfolio report from notebook 07.")
elif segments_file.exists():
    portfolio = pd.read_csv(segments_file, index_col=0)
    print("Loaded risk segments file as fallback input.")
else:
    raise FileNotFoundError(
        "Missing portfolio inputs. Run notebooks/04_unsupervised_risk_segmentation.ipynb "
        "and notebooks/07_portfolio_risk_reporting.ipynb before this notebook."
    )

portfolio.index = portfolio.index.astype(str)

required_columns = ["predicted_pd", "credit_amount", "actual_default"]

missing_required_columns = [col for col in required_columns if col not in portfolio.columns]

if missing_required_columns:
    raise ValueError(f"Missing required columns: {missing_required_columns}")

if "ead_proxy" not in portfolio.columns:
    portfolio["ead_proxy"] = portfolio["credit_amount"]

if "risk_cluster" not in portfolio.columns:
    portfolio["risk_cluster"] = "unknown_cluster"

if "risk_band" not in portfolio.columns:
    def assign_pd_band(pd_value: float) -> str:
        if pd.isna(pd_value):
            return "missing_pd"
        if pd_value < 0.20:
            return "low_risk"
        elif pd_value < 0.40:
            return "moderate_risk"
        elif pd_value < 0.60:
            return "elevated_risk"
        else:
            return "high_risk"
    portfolio["risk_band"] = portfolio["predicted_pd"].apply(assign_pd_band)

if "ensemble_anomaly_flag" not in portfolio.columns:
    portfolio["ensemble_anomaly_flag"] = 0

portfolio.head()

## 4. Base portfolio assumptions

The expected loss framework is:

```text
Expected Loss = PD × LGD × EAD
```

Where:

- **PD** is probability of default.
- **LGD** is loss given default.
- **EAD** is exposure at default.

Because the German Credit dataset does not provide true LGD or EAD, the project uses:

- `predicted_pd` as model-based PD.
- `credit_amount` or `ead_proxy` as exposure proxy.
- A constant base LGD assumption of 45%.

These assumptions are simplified, but they allow the project to connect machine learning outputs with credit risk portfolio logic.

In [ ]:
BASE_LGD = 0.45

portfolio["base_pd"] = portfolio["predicted_pd"].clip(0, 1)
portfolio["base_lgd"] = BASE_LGD
portfolio["base_ead"] = portfolio["ead_proxy"].clip(lower=0)
portfolio["base_expected_loss"] = portfolio["base_pd"] * portfolio["base_lgd"] * portfolio["base_ead"]

base_portfolio_summary = {
    "n_borrowers": int(len(portfolio)),
    "total_ead": float(portfolio["base_ead"].sum()),
    "average_pd": float(portfolio["base_pd"].mean()),
    "observed_default_rate": float(portfolio["actual_default"].mean()),
    "base_lgd": BASE_LGD,
    "base_expected_loss": float(portfolio["base_expected_loss"].sum()),
    "base_expected_loss_rate": float(portfolio["base_expected_loss"].sum() / portfolio["base_ead"].sum()),
}

base_portfolio_summary_df = pd.DataFrame(
    list(base_portfolio_summary.items()),
    columns=["metric", "value"],
)

base_portfolio_summary_df

## 5. Define stress scenarios

The scenarios below are deliberately simple and transparent.

Each scenario applies three types of shocks:

1. **General PD multiplier**: increases the probability of default across the portfolio.
2. **LGD level**: raises the assumed loss given default.
3. **EAD multiplier**: changes the exposure proxy.
4. **Targeted stress add-ons**: extra PD deterioration for high-risk clusters, high-risk bands, and anomalous borrowers.

The targeted add-ons make the stress test more interesting because not every borrower deteriorates in the same way.

In [ ]:
stress_scenarios = pd.DataFrame(
    [
        {
            "scenario": "base",
            "pd_multiplier": 1.00,
            "lgd": 0.45,
            "ead_multiplier": 1.00,
            "high_risk_band_pd_addon": 0.00,
            "high_risk_cluster_pd_addon": 0.00,
            "anomaly_pd_addon": 0.00,
            "description": "Current model estimate with base LGD assumption.",
        },
        {
            "scenario": "mild_stress",
            "pd_multiplier": 1.15,
            "lgd": 0.50,
            "ead_multiplier": 1.00,
            "high_risk_band_pd_addon": 0.02,
            "high_risk_cluster_pd_addon": 0.01,
            "anomaly_pd_addon": 0.01,
            "description": "Moderate deterioration in borrower risk and recovery assumptions.",
        },
        {
            "scenario": "adverse_stress",
            "pd_multiplier": 1.30,
            "lgd": 0.55,
            "ead_multiplier": 1.03,
            "high_risk_band_pd_addon": 0.05,
            "high_risk_cluster_pd_addon": 0.03,
            "anomaly_pd_addon": 0.03,
            "description": "Adverse credit environment with stronger deterioration in risky profiles.",
        },
        {
            "scenario": "severe_stress",
            "pd_multiplier": 1.55,
            "lgd": 0.65,
            "ead_multiplier": 1.05,
            "high_risk_band_pd_addon": 0.08,
            "high_risk_cluster_pd_addon": 0.05,
            "anomaly_pd_addon": 0.05,
            "description": "Severe stress with broad PD deterioration and higher loss severity.",
        },
    ]
)

stress_scenarios

## 6. Identify vulnerable segments

To apply targeted stress, the notebook identifies:

- High-risk bands: `elevated_risk` and `high_risk`.
- High-risk clusters: clusters whose average PD is above the portfolio average.
- Anomalous borrowers: borrowers flagged by the anomaly detection layer.

This creates a stress design where riskier and more unusual profiles receive additional deterioration.

In [ ]:
portfolio_avg_pd = portfolio["base_pd"].mean()

cluster_pd_summary = (
    portfolio
    .groupby("risk_cluster")
    .agg(
        borrowers=("base_pd", "size"),
        avg_pd=("base_pd", "mean"),
        observed_default_rate=("actual_default", "mean"),
        total_ead=("base_ead", "sum"),
        base_expected_loss=("base_expected_loss", "sum"),
    )
    .assign(
        ead_share=lambda x: x["total_ead"] / x["total_ead"].sum(),
        base_expected_loss_share=lambda x: x["base_expected_loss"] / x["base_expected_loss"].sum(),
    )
    .sort_values("avg_pd", ascending=False)
)

high_risk_clusters = cluster_pd_summary[
    cluster_pd_summary["avg_pd"] > portfolio_avg_pd
].index.tolist()

high_risk_bands = ["elevated_risk", "high_risk"]

print(f"Portfolio average PD: {portfolio_avg_pd:.4f}")
print(f"High-risk clusters: {high_risk_clusters}")
print(f"High-risk bands: {high_risk_bands}")

cluster_pd_summary

## 7. Stress testing engine

The function below applies each scenario to every borrower.

For each scenario:

1. Start with base PD.
2. Multiply PD by the scenario-level PD multiplier.
3. Add targeted PD stress for high-risk bands, high-risk clusters, and anomalies.
4. Cap stressed PD at 100%.
5. Apply scenario LGD.
6. Apply scenario EAD multiplier.
7. Calculate stressed expected loss.

In [ ]:
def apply_stress_scenario(
    data: pd.DataFrame,
    scenario_row: pd.Series,
    high_risk_clusters: list,
    high_risk_bands: list,
) -> pd.DataFrame:
    stressed = data.copy()

    scenario_name = scenario_row["scenario"]

    stressed_pd = stressed["base_pd"] * scenario_row["pd_multiplier"]

    stressed_pd += np.where(
        stressed["risk_band"].isin(high_risk_bands),
        scenario_row["high_risk_band_pd_addon"],
        0.0,
    )

    stressed_pd += np.where(
        stressed["risk_cluster"].isin(high_risk_clusters),
        scenario_row["high_risk_cluster_pd_addon"],
        0.0,
    )

    stressed_pd += np.where(
        stressed["ensemble_anomaly_flag"] == 1,
        scenario_row["anomaly_pd_addon"],
        0.0,
    )

    stressed[f"{scenario_name}_pd"] = stressed_pd.clip(0, 1)
    stressed[f"{scenario_name}_lgd"] = float(scenario_row["lgd"])
    stressed[f"{scenario_name}_ead"] = stressed["base_ead"] * float(scenario_row["ead_multiplier"])
    stressed[f"{scenario_name}_expected_loss"] = (
        stressed[f"{scenario_name}_pd"]
        * stressed[f"{scenario_name}_lgd"]
        * stressed[f"{scenario_name}_ead"]
    )

    stressed[f"{scenario_name}_expected_loss_rate"] = (
        stressed[f"{scenario_name}_expected_loss"]
        / stressed[f"{scenario_name}_ead"].replace(0, np.nan)
    )

    return stressed

stress_results = portfolio.copy()

for _, scenario_row in stress_scenarios.iterrows():
    scenario_output = apply_stress_scenario(
        portfolio,
        scenario_row,
        high_risk_clusters=high_risk_clusters,
        high_risk_bands=high_risk_bands,
    )

    scenario_name = scenario_row["scenario"]
    scenario_cols = [
        f"{scenario_name}_pd",
        f"{scenario_name}_lgd",
        f"{scenario_name}_ead",
        f"{scenario_name}_expected_loss",
        f"{scenario_name}_expected_loss_rate",
    ]

    stress_results[scenario_cols] = scenario_output[scenario_cols]

stress_results.head()

## 8. Portfolio-level stress results

This section summarizes the total impact of each scenario.

The key outputs are:

- Total exposure under scenario.
- Average stressed PD.
- Scenario LGD.
- Total expected loss.
- Expected loss rate.
- Increase in expected loss versus base.

In [ ]:
portfolio_stress_summary_rows = []

base_expected_loss = stress_results["base_expected_loss"].sum()

for _, scenario_row in stress_scenarios.iterrows():
    scenario_name = scenario_row["scenario"]

    scenario_total_ead = stress_results[f"{scenario_name}_ead"].sum()
    scenario_total_el = stress_results[f"{scenario_name}_expected_loss"].sum()

    portfolio_stress_summary_rows.append(
        {
            "scenario": scenario_name,
            "description": scenario_row["description"],
            "total_ead": scenario_total_ead,
            "average_pd": stress_results[f"{scenario_name}_pd"].mean(),
            "median_pd": stress_results[f"{scenario_name}_pd"].median(),
            "lgd": scenario_row["lgd"],
            "total_expected_loss": scenario_total_el,
            "expected_loss_rate": scenario_total_el / scenario_total_ead,
            "expected_loss_increase_vs_base": scenario_total_el - base_expected_loss,
            "expected_loss_multiplier_vs_base": scenario_total_el / base_expected_loss,
        }
    )

portfolio_stress_summary = pd.DataFrame(portfolio_stress_summary_rows)

portfolio_stress_summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

portfolio_stress_summary.set_index("scenario")["total_expected_loss"].plot(kind="bar", ax=ax)

ax.set_title("Total Expected Loss by Stress Scenario")
ax.set_xlabel("Scenario")
ax.set_ylabel("Total expected loss")
ax.tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.savefig(REPORTS_FIGURES / "08_total_expected_loss_by_scenario.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

portfolio_stress_summary.set_index("scenario")["expected_loss_rate"].plot(kind="bar", ax=ax)

ax.set_title("Expected Loss Rate by Stress Scenario")
ax.set_xlabel("Scenario")
ax.set_ylabel("Expected loss rate")
ax.tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.savefig(REPORTS_FIGURES / "08_expected_loss_rate_by_scenario.png", dpi=150)
plt.show()

## 9. Stress results by risk band

Risk band analysis shows which supervised PD groups absorb the largest stress impact.

This is useful for portfolio monitoring because a moderate number of high-risk borrowers can dominate stressed loss.

In [ ]:
risk_band_stress_rows = []

for _, scenario_row in stress_scenarios.iterrows():
    scenario_name = scenario_row["scenario"]

    summary = (
        stress_results
        .groupby("risk_band")
        .agg(
            borrowers=("actual_default", "size"),
            total_ead=(f"{scenario_name}_ead", "sum"),
            avg_pd=(f"{scenario_name}_pd", "mean"),
            total_expected_loss=(f"{scenario_name}_expected_loss", "sum"),
            observed_default_rate=("actual_default", "mean"),
        )
        .assign(
            scenario=scenario_name,
            expected_loss_share=lambda x: x["total_expected_loss"] / x["total_expected_loss"].sum(),
            ead_share=lambda x: x["total_ead"] / x["total_ead"].sum(),
        )
        .reset_index()
    )

    risk_band_stress_rows.append(summary)

risk_band_stress_summary = pd.concat(risk_band_stress_rows, axis=0)

risk_band_stress_summary.head()

In [ ]:
risk_band_el_pivot = risk_band_stress_summary.pivot(
    index="risk_band",
    columns="scenario",
    values="total_expected_loss",
)

risk_band_el_pivot

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

risk_band_el_pivot.plot(kind="bar", ax=ax)

ax.set_title("Expected Loss by Risk Band Across Stress Scenarios")
ax.set_xlabel("Risk band")
ax.set_ylabel("Total expected loss")
ax.tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.savefig(REPORTS_FIGURES / "08_expected_loss_by_risk_band_scenarios.png", dpi=150)
plt.show()

## 10. Stress results by risk cluster

This section evaluates how each unsupervised risk segment behaves under stress.

A cluster may become important because it has high PD, high exposure, high LGD sensitivity, or a combination of these factors.

In [ ]:
cluster_stress_rows = []

for _, scenario_row in stress_scenarios.iterrows():
    scenario_name = scenario_row["scenario"]

    summary = (
        stress_results
        .groupby("risk_cluster")
        .agg(
            borrowers=("actual_default", "size"),
            total_ead=(f"{scenario_name}_ead", "sum"),
            avg_pd=(f"{scenario_name}_pd", "mean"),
            total_expected_loss=(f"{scenario_name}_expected_loss", "sum"),
            observed_default_rate=("actual_default", "mean"),
            anomaly_rate=("ensemble_anomaly_flag", "mean"),
        )
        .assign(
            scenario=scenario_name,
            expected_loss_share=lambda x: x["total_expected_loss"] / x["total_expected_loss"].sum(),
            ead_share=lambda x: x["total_ead"] / x["total_ead"].sum(),
        )
        .reset_index()
    )

    cluster_stress_rows.append(summary)

cluster_stress_summary = pd.concat(cluster_stress_rows, axis=0)

cluster_stress_summary.head()

In [ ]:
cluster_el_pivot = cluster_stress_summary.pivot(
    index="risk_cluster",
    columns="scenario",
    values="total_expected_loss",
)

cluster_el_pivot

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

cluster_el_pivot.plot(kind="bar", ax=ax)

ax.set_title("Expected Loss by Risk Cluster Across Stress Scenarios")
ax.set_xlabel("Risk cluster")
ax.set_ylabel("Total expected loss")
ax.tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.savefig(REPORTS_FIGURES / "08_expected_loss_by_cluster_scenarios.png", dpi=150)
plt.show()

## 11. Anomaly stress impact

This section compares normal and anomalous borrower profiles under each scenario.

Anomalies receive an additional PD shock in the scenario design because unusual borrower profiles may be harder to assess and monitor.

In [ ]:
anomaly_stress_rows = []

for _, scenario_row in stress_scenarios.iterrows():
    scenario_name = scenario_row["scenario"]

    summary = (
        stress_results
        .groupby("ensemble_anomaly_flag")
        .agg(
            borrowers=("actual_default", "size"),
            total_ead=(f"{scenario_name}_ead", "sum"),
            avg_pd=(f"{scenario_name}_pd", "mean"),
            total_expected_loss=(f"{scenario_name}_expected_loss", "sum"),
            observed_default_rate=("actual_default", "mean"),
        )
        .assign(
            scenario=scenario_name,
            expected_loss_share=lambda x: x["total_expected_loss"] / x["total_expected_loss"].sum(),
            ead_share=lambda x: x["total_ead"] / x["total_ead"].sum(),
        )
        .reset_index()
    )

    summary["profile_type"] = np.where(
        summary["ensemble_anomaly_flag"] == 1,
        "anomalous_profile",
        "normal_profile",
    )

    anomaly_stress_rows.append(summary)

anomaly_stress_summary = pd.concat(anomaly_stress_rows, axis=0)

anomaly_stress_summary

In [ ]:
anomaly_el_pivot = anomaly_stress_summary.pivot(
    index="profile_type",
    columns="scenario",
    values="total_expected_loss",
)

anomaly_el_pivot

## 12. Sensitivity analysis: PD and LGD grid

Scenario analysis uses selected assumptions. Sensitivity analysis asks a broader question: how does total expected loss change across many combinations of PD and LGD shocks?

This grid applies:

- PD multipliers from 1.00 to 1.75.
- LGD values from 35% to 75%.

The output helps identify whether the portfolio is more sensitive to default frequency or loss severity.

In [ ]:
pd_multipliers = np.arange(1.00, 1.80, 0.10)
lgd_values = np.arange(0.35, 0.80, 0.05)

sensitivity_rows = []

for pd_multiplier in pd_multipliers:
    for lgd in lgd_values:
        stressed_pd = (portfolio["base_pd"] * pd_multiplier).clip(0, 1)
        stressed_el = stressed_pd * lgd * portfolio["base_ead"]

        sensitivity_rows.append(
            {
                "pd_multiplier": round(float(pd_multiplier), 2),
                "lgd": round(float(lgd), 2),
                "total_expected_loss": float(stressed_el.sum()),
                "expected_loss_rate": float(stressed_el.sum() / portfolio["base_ead"].sum()),
                "expected_loss_multiplier_vs_base": float(stressed_el.sum() / base_expected_loss),
            }
        )

sensitivity_grid = pd.DataFrame(sensitivity_rows)

sensitivity_grid.head()

In [ ]:
sensitivity_pivot = sensitivity_grid.pivot(
    index="lgd",
    columns="pd_multiplier",
    values="expected_loss_multiplier_vs_base",
)

sensitivity_pivot

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

im = ax.imshow(sensitivity_pivot, aspect="auto")

ax.set_title("Expected Loss Multiplier Sensitivity: PD Multiplier × LGD")
ax.set_xlabel("PD multiplier")
ax.set_ylabel("LGD")

ax.set_xticks(range(len(sensitivity_pivot.columns)))
ax.set_xticklabels(sensitivity_pivot.columns, rotation=45)

ax.set_yticks(range(len(sensitivity_pivot.index)))
ax.set_yticklabels([f"{value:.2f}" for value in sensitivity_pivot.index])

fig.colorbar(im, ax=ax, label="EL multiplier vs base")

plt.tight_layout()
plt.savefig(REPORTS_FIGURES / "08_pd_lgd_sensitivity_heatmap.png", dpi=150)
plt.show()

## 13. Top borrowers under severe stress

This section identifies borrowers with the highest expected loss under the severe stress scenario.

This table is useful for review prioritization. It should not be interpreted as an automatic rejection or collection list.

In [ ]:
severe_scenario = "severe_stress"

top_severe_stress_borrowers = (
    stress_results
    .assign(
        severe_expected_loss=stress_results[f"{severe_scenario}_expected_loss"],
        severe_pd=stress_results[f"{severe_scenario}_pd"],
        severe_ead=stress_results[f"{severe_scenario}_ead"],
    )
    .sort_values("severe_expected_loss", ascending=False)
    .head(25)
)

review_columns = [
    "actual_default",
    "risk_band",
    "risk_cluster",
    "ensemble_anomaly_flag",
    "credit_amount",
    "duration_months",
    "age_years",
    "base_pd",
    "severe_pd",
    "base_expected_loss",
    "severe_expected_loss",
]

available_review_columns = [col for col in review_columns if col in top_severe_stress_borrowers.columns]

top_severe_stress_borrowers[available_review_columns]

## 14. Stress testing executive narrative

This section generates a concise written summary of the stress testing results.

The text can be reused in the README, the markdown report, or the future Streamlit dashboard.

In [ ]:
base_row = portfolio_stress_summary.loc[
    portfolio_stress_summary["scenario"] == "base"
].iloc[0]

severe_row = portfolio_stress_summary.loc[
    portfolio_stress_summary["scenario"] == "severe_stress"
].iloc[0]

adverse_row = portfolio_stress_summary.loc[
    portfolio_stress_summary["scenario"] == "adverse_stress"
].iloc[0]

largest_severe_cluster = (
    cluster_stress_summary
    .loc[cluster_stress_summary["scenario"] == "severe_stress"]
    .sort_values("total_expected_loss", ascending=False)
    .iloc[0]
)

largest_severe_band = (
    risk_band_stress_summary
    .loc[risk_band_stress_summary["scenario"] == "severe_stress"]
    .sort_values("total_expected_loss", ascending=False)
    .iloc[0]
)

stress_narrative = f"""
# Stress Testing Report

The base scenario estimates total expected loss at {base_row['total_expected_loss']:,.2f}, equivalent to an expected loss rate of {base_row['expected_loss_rate']:.2%}.

Under the adverse scenario, total expected loss rises to {adverse_row['total_expected_loss']:,.2f}, representing a multiplier of {adverse_row['expected_loss_multiplier_vs_base']:.2f} times the base expected loss.

Under the severe scenario, total expected loss rises to {severe_row['total_expected_loss']:,.2f}, equivalent to an expected loss rate of {severe_row['expected_loss_rate']:.2%}. This represents a multiplier of {severe_row['expected_loss_multiplier_vs_base']:.2f} times the base expected loss.

In the severe scenario, the largest cluster-level contribution to expected loss comes from risk cluster `{largest_severe_cluster['risk_cluster']}`, with an expected loss share of {largest_severe_cluster['expected_loss_share']:.2%}. The largest risk-band contribution comes from `{largest_severe_band['risk_band']}`, with an expected loss share of {largest_severe_band['expected_loss_share']:.2%}.

These results indicate that the portfolio should be monitored through both broad credit deterioration assumptions and targeted deterioration in high-risk segments, anomalous profiles, and high-exposure borrowers.
"""

print(stress_narrative)

## 15. Safe markdown export helper

`pandas.to_markdown()` requires the optional `tabulate` package. The helper below uses markdown tables when available and falls back to plain text tables if `tabulate` is not installed.

If you want cleaner markdown tables, install:

```bash
pip install tabulate
```

In [ ]:
def dataframe_to_markdown_safe(df: pd.DataFrame, index: bool = True) -> str:
    try:
        return df.to_markdown(index=index)
    except ImportError:
        return df.to_string(index=index)

## 16. Save stress testing outputs

This section saves:

- Borrower-level stress results.
- Portfolio scenario summary.
- Risk band stress summary.
- Cluster stress summary.
- Anomaly stress summary.
- PD-LGD sensitivity grid.
- Top borrowers under severe stress.
- Markdown stress testing report.
- Metadata for reproducibility.

In [ ]:
stress_results.to_csv(DATA_PROCESSED / "credit_risk_stress_results.csv", index=True)

portfolio_stress_summary.to_csv(REPORTS_TABLES / "08_portfolio_stress_summary.csv", index=False)
risk_band_stress_summary.to_csv(REPORTS_TABLES / "08_risk_band_stress_summary.csv", index=False)
cluster_stress_summary.to_csv(REPORTS_TABLES / "08_cluster_stress_summary.csv", index=False)
anomaly_stress_summary.to_csv(REPORTS_TABLES / "08_anomaly_stress_summary.csv", index=False)
sensitivity_grid.to_csv(REPORTS_TABLES / "08_pd_lgd_sensitivity_grid.csv", index=False)
top_severe_stress_borrowers[available_review_columns].to_csv(
    REPORTS_TABLES / "08_top_severe_stress_borrowers.csv",
    index=True,
)
stress_scenarios.to_csv(REPORTS_TABLES / "08_stress_scenarios.csv", index=False)

stress_report_path = REPORTS_DIR / "credit_risk_stress_testing_report.md"

with open(stress_report_path, "w", encoding="utf-8") as f:
    f.write(stress_narrative)
    f.write("\n\n## Stress Scenarios\n\n")
    f.write(dataframe_to_markdown_safe(stress_scenarios, index=False))
    f.write("\n\n## Portfolio Stress Summary\n\n")
    f.write(dataframe_to_markdown_safe(portfolio_stress_summary, index=False))
    f.write("\n\n## Risk Band Stress Summary\n\n")
    f.write(dataframe_to_markdown_safe(risk_band_stress_summary, index=False))
    f.write("\n\n## Cluster Stress Summary\n\n")
    f.write(dataframe_to_markdown_safe(cluster_stress_summary, index=False))
    f.write("\n\n## Top Severe Stress Borrowers\n\n")
    f.write(dataframe_to_markdown_safe(top_severe_stress_borrowers[available_review_columns], index=True))

stress_metadata = {
    "report_layer": "stress_testing",
    "base_lgd": float(BASE_LGD),
    "n_scenarios": int(len(stress_scenarios)),
    "scenarios": stress_scenarios["scenario"].tolist(),
    "n_borrowers": int(len(stress_results)),
    "base_expected_loss": float(base_row["total_expected_loss"]),
    "adverse_expected_loss": float(adverse_row["total_expected_loss"]),
    "severe_expected_loss": float(severe_row["total_expected_loss"]),
    "adverse_expected_loss_multiplier_vs_base": float(adverse_row["expected_loss_multiplier_vs_base"]),
    "severe_expected_loss_multiplier_vs_base": float(severe_row["expected_loss_multiplier_vs_base"]),
    "high_risk_clusters": [str(cluster) for cluster in high_risk_clusters],
    "high_risk_bands": high_risk_bands,
}

with open(MODELS_DIR / "stress_testing_metadata.json", "w") as f:
    json.dump(stress_metadata, f, indent=4)

print(f"Saved borrower-level stress results to: {DATA_PROCESSED / 'credit_risk_stress_results.csv'}")
print(f"Saved markdown stress report to: {stress_report_path}")
print("Saved stress testing tables, figures, and metadata.")

## 17. Analytical conclusions

This notebook added the stress testing layer to the Credit Risk Intelligence Lab.

The project now has a complete first version of a credit risk analytics workflow:

1. Understand the portfolio through exploratory analysis.
2. Engineer financial risk features.
3. Estimate probability of default.
4. Discover hidden borrower segments.
5. Detect anomalous credit profiles.
6. Explain model behavior.
7. Report portfolio-level risk.
8. Stress test expected loss under adverse conditions.

Key outputs:

- Stress scenario definitions.
- Borrower-level stressed PD, LGD, EAD, and expected loss.
- Portfolio-level stress summary.
- Stress results by risk band.
- Stress results by risk cluster.
- Anomaly stress impact.
- PD-LGD sensitivity grid.
- Severe stress borrower review table.
- Markdown stress testing report.

At this point, the core notebook pipeline is complete. The next project layer should not necessarily be another notebook. A better next step is to build:

```text
app/streamlit_app.py
```

The dashboard can present the portfolio risk KPIs, clusters, anomalies, interpretability summaries, and stress testing results in an interactive format.